In [8]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from qdrant_client import QdrantClient
from openai import OpenAI
load_dotenv("backend/.env", override=True)
print(os.getenv("QDRANT_URL"))
qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)
print(qdrant_client.get_collections())
# client = OpenAI(
#     api_key=os.getenv("YOUR_EURI_API_KEY"),
#     base_url="https://api.euron.one/api/v1/euri"
# )

client = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="The quick brown fox jumps over the lazy dog."
)
print(response.data[0].embedding[:5])  # First 5 dimensions

https://e24b3dd5-990d-4b33-b73e-9b3ddcdae69d.sa-east-1-0.aws.cloud.qdrant.io
collections=[]


AttributeError: 'ChatGoogleGenerativeAI' object has no attribute 'embeddings'

In [2]:
import os
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_qdrant import QdrantVectorStore

load_dotenv("backend/.env", override=True)

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)



loader = TextLoader("backend/app/rag/documents/payment_policy.txt")

documents = loader.load()

#print(documents)

# print(len(documents))
# print(documents[0].page_content)
# print(documents[0].metadata)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=[".", "\n", " ", ""]
)


chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
     print(f"\n--- Chunk {i + 1} ---")
     print(chunk.page_content)


vector = embeddings.embed_query(
    chunks[0].page_content
)

#print("Vector length:", len(vector))
#print("First 10 values:", vector[:10])


vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    collection_name="payment_policy",
)

# print("Documents stored in Qdrant")



C:\Users\omkar\AppData\Local\Temp\ipykernel_12404\3101614848.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Number of chunks: 5

--- Chunk 1 ---
Payment Reconciliation Policy

Payments are reconciled against the expected invoice amount

--- Chunk 2 ---
.

A payment is considered matched when the payment amount
equals the expected invoice amount

--- Chunk 3 ---
.

If the payment amount differs from the expected amount,
the payment is marked as a discrepancy

--- Chunk 4 ---
.

Discrepancies can be caused by incorrect payment amounts,
duplicate payments, or missing payments

--- Chunk 5 ---
.

Refund requests must be submitted within 30 days.


In [3]:
query = "What is the deadline for requesting a refund?"

results = vector_store.similarity_search(
    query,
    k=3
)

print("Number of results:", len(results))

for i, doc in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(doc.page_content)

Number of results: 3

--- Result 1 ---
.

Refund requests must be submitted within 30 days.

--- Result 2 ---
Payment Reconciliation Policy

Payments are reconciled against the expected invoice amount

--- Result 3 ---
.

A payment is considered matched when the payment amount
equals the expected invoice amount


In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

context = "\n\n".join(
    doc.page_content for doc in results
)

question = "What is the deadline for requesting a refund?"

prompt = f"""
Answer the user's question using only the information in the context.

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'Based on the provided context, refund requests must be submitted within 30 days.', 'extras': {'signature': 'EosHCogHARFNMg8AaMQMPjaSm4RD/lA3kjVETwFeyLjU4jTR2hhkEWTmbnYnis9QT4zJh3DcTs2Ilip/QdZUnW79TdW+EcDdWt+twPU3b9p0s6aKaa8UPKMKNvAegizvAIa8m3QuIp23j+b4EECuKJ+UkRzOGqaq34bjmofQQ0nJWOJEwMc7G16YubPrwrihbDFr+v9Q8T1bOaeZq4zmUsuzFhBBM5RPUuFgmk5Z5jmx0DM0V3+MoiAlYEedCtCFq7oNGeovKv1CxCReJvLJ6loMCqv/Ko3w5VPEPUJPnOE7ubTX2yIchzh6eiX8fkK+HHH2KJvIbOlMItPuqAp9E7Sz9g9DjbwLEmceRn5xJaD7bbbKy1MmZPCdLFTLGrRKlLOf/BTslcWOiQerMXXppYlVEaQ2p40/TGJn+5ZEql7TZQ5B3rOIzq5kLa0fRWK8euKFY3ooguL5YEtBcUPhJ0zu0SHPInnBXVfFI1WRg899hmZ+TXRqtrSw8iw1ioRUiSnnCpEmN2bdSyJADSw45tz1hZeqr9ftDKzUQlMd42EgE7DFV67Tm/0kJuvew9iFDq6xCLiuyPfn4KQdT+DtDXGjcxOyyTBkOzUnqsL9FnQxROucDltG17rx1eWc6pwO6KcpcjsbsvkNQTSEAuouVx0qDxRSOsv0zWUvqLMMnCojYLu8wfetzhkklXm8ebyMY22RWdy41sPU57a6MS/297V6poh5GxTSzln+1NpigQGIpCYu8l7CSBnnGNMpqtAqxgNl4TqWhFrt4EE75uWadYP4Y2pXrexd/qs6XpBh5cbi8E6L0oQXUF+vNJO1nO3YjqNCXQYShqHKVwAK/ZhmQWIvaGuCbysaf

In [ ]:
import os

from dotenv import load_dotenv

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


load_dotenv("backend/.env", override=True)


# -------------------------
# Configuration
# -------------------------

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")


# -------------------------
# Gemini
# -------------------------

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    api_key=os.getenv("GOOGLE_API_KEY")
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
)


# -------------------------
# Qdrant Cloud
# -------------------------

qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)


# -------------------------
# Documents
# -------------------------

documents = [
    Document(
        page_content="""
        Qdrant is a vector database designed for
        similarity search and AI applications.
        """
    ),
    Document(
        page_content="""
        LangChain is a framework for developing
        applications powered by language models.
        """
    ),
]


# -------------------------
# Chunking
# -------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(documents)


# -------------------------
# Store embeddings in Qdrant
# -------------------------

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    collection_name="simple_rag",
)


# -------------------------
# Retriever
# -------------------------

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)


# -------------------------
# RAG query
# -------------------------

question = "What is Qdrant?"

docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content
    for doc in docs
)


prompt = f"""
You are a helpful assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

If the answer is not available in the context,
say "I don't know based on the provided documents."
"""

response = llm.invoke(prompt)

print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


GoogleModelNotFoundError: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}

In [1]:
%pip install langchain_qdrant

  Using cached langchain_qdrant-1.1.0-py3-none-any.whl.metadata (2.0 kB)
Using cached langchain_qdrant-1.1.0-py3-none-any.whl (24 kB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
#pip install qdrant-client
%pip install openai

  Using cached openai-3.6.0-py3-none-any.whl.metadata (41 kB)
  Using cached httpx2-2.12.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached jiter-0.16.0-cp314-cp314-win_amd64.whl.metadata (5.3 kB)
  Using cached httpcore2-2.12.0-py3-none-any.whl.metadata (25 kB)
  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
Using cached openai-3.6.0-py3-none-any.whl (1.7 MB)
Using cached httpx2-2.12.0-py3-none-any.whl (95 kB)
Using cached httpcore2-2.12.0-py3-none-any.whl (83 kB)
Using cached jiter-0.16.0-cp314-cp314-win_amd64.whl (198 kB)
Using cached truststore-0.10.4-py3-none-any.whl (18 kB)

   ---------------- ----------------------- 2/5 [httpcore2]
   ------------------------ --------------- 3/5 [httpx2]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/